<p class="lead">This supplement gives a programmer everything needed to read and run the R code in the rest of the BGM-411 study material. It is the R counterpart of the Python toolkit supplement, and it also covers the ground of the course's first laboratory, which the official plan sets in R: vectors, matrices, data frames and lists, and the indexing patterns that every later unit relies on. If you have never written R, work through every chunk; if you know R, skim to Section 6 (the modelling workflow).</p>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>Run R in Google Colab (R runtime) and explain how an R-Markdown notebook mixes prose, code and output.</li>
<li>Use R's core data structures — atomic vectors, matrices, lists and the <code>data.frame</code> — and index them four ways.</li>
<li>Vectorise a computation instead of looping, and read R's recycling rules.</li>
<li>Load a dataset from the <code>mlbench</code> package, explore it, and plot it with base graphics and <code>ggplot2</code>.</li>
<li>Carry out a complete supervised-learning workflow in R: split, train, predict, evaluate — the template for Units 2–9.</li>
<li>Make an analysis reproducible with <code>set.seed</code> and good habits.</li>
</ul></div>

<div class="box bridge"><h4>How to read this notebook</h4>
<p>In Google Colab choose <em>Runtime → Change runtime type → R</em>, or open the <code>.ipynb</code> (its kernel is already R). Each grey block is R code; the block beneath it is the output exactly as R produced it. The only packages used across the course — <code>MASS</code>, <code>class</code>, <code>e1071</code>, <code>rpart</code>, <code>randomForest</code>, <code>cluster</code>, <code>nnet</code>, <code>neuralnet</code>, <code>fastICA</code>, <code>ggplot2</code>, <code>mlbench</code>, <code>pROC</code> — are pre-installed in this environment; in a fresh Colab run <code>install.packages(c("e1071","randomForest","mlbench","pROC"))</code> once.</p></div>

## 1. Why R, and what a notebook is

R is a language built for data: statistics, modelling and graphics are in the language and its standard library rather than bolted on.<sup class="c"><a href="#r1">[1]</a></sup> For this course R has a second advantage — it is the language the official laboratory plan uses, so the notebooks double as lab material; a broader gentle introduction is *R for Data Science*.<sup class="c"><a href="#r8">[8]</a></sup> An **R-Markdown notebook** interleaves prose and executable R; when "knitted" it runs every chunk top to bottom and freezes the results next to the code,<sup class="c"><a href="#r2">[2]</a></sup> which is exactly how these pages were produced. The first rule of notebooks is therefore the same as in any language: **a notebook is only trustworthy if it runs top to bottom in a fresh session**, because out-of-order execution hides bugs.



In [ ]:
# The libraries used across the course. In a fresh Colab R runtime, install once (see intro).
suppressPackageStartupMessages({
  library(ggplot2); library(mlbench)
})
set.seed(0)                      # reproducibility: fixes every random draw below
R.version.string



The printout is the R version these notes were built with. `set.seed(0)` makes every later random split reproducible — the single most important habit of Section 7.

## 2. The vocabulary of R: vectors, matrices, lists, data frames

R's atomic object is the **vector**; even a single number is a length-1 vector. Vectors are typed (double, integer, logical, character) and operations are *vectorised* — they apply element by element, with no loop.



In [ ]:
x <- c(2, 4, 6, 8, 10)           # numeric vector;  <-  is assignment
x * 10                            # vectorised: multiplies every element
x[x > 5]                          # logical indexing: elements greater than 5
c(mean = mean(x), sd = sd(x), n = length(x))   # named vector of summaries



Each line returns a vector: the elementwise product, the subset selected by a logical mask, and a named vector of summary statistics. Four indexing styles cover almost everything you will do: by positive position (`x[2:3]`), by negative position to drop (`x[-1]`), by logical mask (`x[x > 5]`), and by name (`x["mean"]`).



In [ ]:
m <- matrix(1:6, nrow = 2)                    # a 2x3 matrix, filled by column
lst <- list(name = "aspirin", mw = 180.2, ok = TRUE)   # a list holds mixed types
df <- data.frame(id = 1:3, dose = c(10, 20, 40), responder = c(TRUE, FALSE, TRUE))
cat("matrix dims:", dim(m), "| list$mw:", lst$mw, "\n")
df



A **matrix** is a vector with a 2-D shape (one type throughout); a **list** is an ordered, named, heterogeneous container accessed with `$` or `[[ ]]`; and a **`data.frame`** is a list of equal-length columns — R's table, the object every dataset and model consumes. The `data.frame` is to R what the `DataFrame` is to pandas.

## 3. Indexing a data frame



In [ ]:
# Rows by condition, columns by name: df[rows, cols]
df[df$responder, c("id", "dose")]        # responders only, two columns
df$dose[df$dose >= 20]                    # a single column, then a mask on it
subset(df, dose < 40, select = dose)      # the readable 'subset' helper



The three cells show the universal pattern `df[rows, columns]`: a logical condition selects rows, a character vector selects columns, and `$` pulls one column as a vector. Mastering this one pattern is 80 % of data manipulation in base R.

## 4. Vectorise, do not loop



In [ ]:
v <- runif(1e5)                           # 100,000 random numbers in [0,1]
t_loop <- system.time({ s <- 0; for (i in seq_along(v)) s <- s + v[i] })["elapsed"]
t_vec  <- system.time({ s2 <- sum(v) })["elapsed"]
cat(sprintf("loop: %.4fs   vectorised sum(): %.5fs   same result: %s\n",
            t_loop, t_vec, isTRUE(all.equal(s, s2))))



The vectorised `sum()` gives the identical result far faster than the explicit loop, because the iteration happens in compiled C. The lesson transfers everywhere in R: prefer a vectorised function (`sum`, `rowMeans`, `apply`, `tapply`) to a hand-written loop.<sup class="c"><a href="#r7">[7]</a></sup>

## 5. A dataset, and two ways to plot it

The recurring dataset of this course is the **Wisconsin breast-cancer** data, 569 fine-needle aspirates described by nuclear-shape features and labelled benign or malignant.<sup class="c"><a href="#r3">[3]</a></sup> The `mlbench` package<sup class="c"><a href="#r4">[4]</a></sup> ships a version, so nothing is downloaded.



In [ ]:
data(BreastCancer, package = "mlbench")
bc <- BreastCancer
bc <- bc[, -1]                                   # drop the Id column
bc <- bc[complete.cases(bc), ]                   # drop the 16 rows with missing values
bc[, 1:9] <- lapply(bc[, 1:9], function(z) as.numeric(as.character(z)))  # ordinal -> numeric
cat("rows x cols:", nrow(bc), "x", ncol(bc), "| classes:\n")
table(bc$Class)



The table shows the class counts (benign vs malignant) after dropping the identifier and the 16 incomplete records — the kind of small curation Supplement S1 treats properly.



In [ ]:
par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))
boxplot(Cl.thickness ~ Class, data = bc, col = c("#2b5d9b", "#d2691e"),
        xlab = "class", ylab = "clump thickness", main = "Base graphics")
plot(bc$Cell.size, bc$Cell.shape, col = ifelse(bc$Class == "malignant", "#d2691e", "#2b5d9b"),
     pch = 19, cex = 0.5, xlab = "cell size", ylab = "cell shape", main = "malignant (orange)")



<p class="caption"><b>Figure 1.</b> Base-R graphics: malignant aspirates (orange) have systematically larger clump thickness and cluster at high cell size and shape — the separability that makes the problem learnable in Unit 2 onward.</p>



In [ ]:
ggplot(bc, aes(Cell.size, Cell.shape, colour = Class)) +
  geom_jitter(width = 0.3, height = 0.3, alpha = 0.6, size = 1) +
  scale_colour_manual(values = c(benign = "#2b5d9b", malignant = "#d2691e")) +
  labs(title = "The same data with ggplot2", x = "cell size", y = "cell shape") +
  theme_minimal(base_size = 11)



<p class="caption"><b>Figure 2.</b> The same relationship drawn with <code>ggplot2</code>,<sup class="c"><a href="#r5">[5]</a></sup> which builds a plot as a sum of layers (<code>geom_*</code>) and mappings (<code>aes</code>). Jitter spreads the overlapping integer scores so density is visible. Both graphics systems appear in the course; use whichever is clearer for the point.</p>

## 6. A complete supervised-learning workflow

Every modelling unit follows the same five steps: split the data, train on the training part, predict on the held-out part, and evaluate. The cell does all five with k-nearest neighbours,<sup class="c"><a href="#r6">[6]</a></sup> the simplest classifier, so the *workflow* — not the model — is the point.



In [ ]:
library(class)
feat <- bc[, 1:9]; y <- bc$Class
n <- nrow(bc); set.seed(0)
train <- sample(n, round(0.7 * n))               # 70% train / 30% test split by row index
sc <- scale(feat)                                 # standardise: k-NN needs comparable scales
pred <- knn(sc[train, ], sc[-train, ], y[train], k = 5)
acc <- mean(pred == y[-train])
cm <- table(truth = y[-train], predicted = pred)
cat(sprintf("test-set accuracy: %.3f  on %d held-out cases\n", acc, length(y) - length(train)))
cm



The confusion matrix cross-tabulates truth against prediction on cases the model never saw; the accuracy above it is the fraction on the diagonal. Supplement S1 explains why this hold-out estimate — not the training accuracy — is the honest one, and introduces the richer metrics (sensitivity, specificity, ROC/AUC) the medical units need.



In [ ]:
ks <- c(1, 3, 5, 9, 15, 25, 41)
accs <- sapply(ks, function(k) mean(knn(sc[train,], sc[-train,], y[train], k = k) == y[-train]))
plot(ks, accs, type = "b", pch = 19, col = "#2b5d9b", xlab = "k (neighbours)",
     ylab = "test accuracy", main = "Choosing k for k-NN")



<p class="caption"><b>Figure 3.</b> Test accuracy against the number of neighbours k. Very small k overfits the training noise and very large k oversmooths; a middle value is best — the bias–variance trade-off that Supplement S1 formalises.</p>

## 7. Reproducibility and good habits



In [ ]:
set.seed(42); a <- sample(5)
set.seed(42); b <- sample(5)
cat("same seed -> identical draw:", identical(a, b), "\n")
sessionInfo()$R.version$version.string



Fixing the seed makes a random analysis repeatable to the exact draw, as the identical vectors show. The habits that make an R analysis trustworthy: set a seed once at the top; never rely on objects created out of order; keep raw data read-only; load packages at the top; and record the session (`sessionInfo()`) so a reader knows the versions. These are the R form of the reproducibility rules the course returns to in Unit 9.

## 8. Summary

- A notebook must run top to bottom in a fresh session; `set.seed` makes it reproducible.
- R's objects: typed vectors (operations vectorise), matrices (one type, 2-D), lists (mixed, named), data frames (the table every model consumes).
- Index with `df[rows, cols]`: positions, negatives, logical masks or names.
- Prefer vectorised functions to loops.
- `mlbench` supplies the breast-cancer data; base graphics and `ggplot2` both plot it.
- The supervised workflow is always split → train → predict → evaluate on held-out data; the confusion matrix and accuracy summarise a classifier.

## 9. Self-assessment

<details><summary>1. Why must a notebook run top to bottom in a fresh session?</summary>Because executing chunks out of order can leave variables in a state the code does not reproduce on a clean run, hiding bugs and making results irreproducible. Knitting (as here) enforces a clean top-to-bottom run.</details>
<details><summary>2. What does <code>x[x > 5]</code> return and what is the mechanism?</summary>The elements of <code>x</code> greater than 5. <code>x > 5</code> builds a logical vector; indexing with it keeps the <code>TRUE</code> positions (logical/boolean indexing).</details>
<details><summary>3. Difference between a matrix, a list and a data frame?</summary>A matrix is a single-type 2-D array; a list is an ordered, named, mixed-type container; a data frame is a list of equal-length columns (possibly different types) — R's table.</details>
<details><summary>4. Why is <code>sum(v)</code> much faster than a <code>for</code> loop summing <code>v</code>?</summary>Vectorised functions run their iteration in compiled C, avoiding the interpreter overhead of an R-level loop.</details>
<details><summary>5. In <code>df[df$responder, c("id","dose")]</code>, what selects rows and what selects columns?</summary>The logical vector <code>df$responder</code> selects rows; the character vector <code>c("id","dose")</code> selects columns, in the pattern <code>df[rows, cols]</code>.</details>
<details><summary>6. Why standardise the features before k-NN?</summary>k-NN uses Euclidean distance, which is dominated by features on larger numeric scales; standardising (<code>scale</code>) puts features on comparable scales so all contribute.</details>
<details><summary>7. What does the confusion matrix show that a single accuracy number hides?</summary>The breakdown of errors by type — how many malignant cases were missed (false negatives) versus benign called malignant (false positives) — which matters because the two errors have different clinical costs.</details>
<details><summary>8. Why evaluate on <code>y[-train]</code> rather than on the training rows?</summary>Training accuracy is optimistically biased because the model has seen those cases; the held-out set estimates performance on unseen data (Supplement S1).</details>
<details><summary>9. What is the effect of k in k-NN, per Figure 3?</summary>Small k overfits (high variance); large k oversmooths (high bias); an intermediate k generalises best.</details>
<details><summary>10. What two things does <code>set.seed(42)</code> before <code>sample()</code> guarantee?</summary>That the random draw is reproducible — the same seed yields the identical sequence — so the split and any stochastic model can be reproduced exactly.</details>
<details><summary>11. How do you drop a column by name versus by position?</summary>By position with a negative index (<code>bc[, -1]</code>); by name by selecting the complement (<code>bc[, setdiff(names(bc), "Id")]</code>).</details>
<details><summary>12. What belongs at the very top of a reproducible R notebook?</summary>Library loads and a single <code>set.seed</code>, before any analysis — so dependencies and randomness are fixed from the start.</details>

## Glossary

<dl class="gloss">
<dt>Vector</dt><dd>R's atomic object: a typed, 1-D sequence; operations apply elementwise (vectorised).</dd>
<dt>Matrix</dt><dd>A vector with a 2-D shape; all elements share one type.</dd>
<dt>List</dt><dd>An ordered, named container of mixed-type elements, accessed with <code>$</code> or <code>[[ ]]</code>.</dd>
<dt>data.frame</dt><dd>A list of equal-length columns; R's table and the input to every model.</dd>
<dt>Vectorisation</dt><dd>Applying an operation to a whole vector at once, in compiled code, instead of looping.</dd>
<dt>Logical indexing</dt><dd>Selecting elements/rows with a <code>TRUE</code>/<code>FALSE</code> mask.</dd>
<dt>Confusion matrix</dt><dd>A cross-tabulation of true vs predicted classes on a test set.</dd>
<dt>Seed</dt><dd>The initial state of the random-number generator; fixing it makes random results reproducible.</dd>
<dt>R Markdown</dt><dd>A document format interleaving prose and R chunks that is knitted into HTML with executed output.</dd>
</dl>

## References and further reading

<ol class="refs">
<li id="r1">R Core Team. R: A Language and Environment for Statistical Computing. R Foundation for Statistical Computing, Vienna. <a href="https://www.R-project.org/">https://www.R-project.org/</a> (software; no DOI)</li>
<li id="r2">Xie Y, Allaire JJ, Grolemund G. <em>R Markdown: The Definitive Guide</em>. Chapman &amp; Hall/CRC; 2018. <a href="https://doi.org/10.1201/9781138359444">doi:10.1201/9781138359444</a></li>
<li id="r3">Street WN, Wolberg WH, Mangasarian OL. Nuclear feature extraction for breast tumor diagnosis. <em>Proc SPIE</em>. 1993;1905:861–870. <a href="https://doi.org/10.1117/12.148698">doi:10.1117/12.148698</a></li>
<li id="r4">Leisch F, Dimitriadou E. mlbench: Machine Learning Benchmark Problems. R package. <a href="https://doi.org/10.32614/CRAN.package.mlbench">doi:10.32614/CRAN.package.mlbench</a></li>
<li id="r5">Wickham H. <em>ggplot2: Elegant Graphics for Data Analysis</em>. 2nd ed. Springer; 2016. <a href="https://doi.org/10.1007/978-3-319-24277-4">doi:10.1007/978-3-319-24277-4</a></li>
<li id="r6">Venables WN, Ripley BD. <em>Modern Applied Statistics with S</em>. 4th ed. Springer; 2002. <a href="https://doi.org/10.1007/978-0-387-21706-2">doi:10.1007/978-0-387-21706-2</a></li>
<li id="r7">Wickham H. <em>Advanced R</em>. 2nd ed. Chapman &amp; Hall/CRC; 2019. <a href="https://doi.org/10.1201/9781351201315">doi:10.1201/9781351201315</a></li>
<li id="r8">Wickham H, Grolemund G. <em>R for Data Science</em>. O'Reilly; 2017. <a href="https://r4ds.had.co.nz/">https://r4ds.had.co.nz/</a> (freely available; no DOI)</li>
</ol>

<h4>Textbooks &amp; resources</h4>
<ul>
<li>James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2nd ed. Springer; 2021 — the R companion to the whole course; free PDF at statlearning.com.</li>
<li>Venables &amp; Ripley, <em>Modern Applied Statistics with S</em> (MASS) — the reference for the R modelling functions used here.</li>
<li>Xie, <em>R Markdown: The Definitive Guide</em> and the RStudio/Posit chedatsheets — for the notebook format.</li>
</ul>
